<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/Global_Job_Category_Imputation_Labor_Market_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Global-Job-Category-Imputation-Labor-Market-Intelligence

Find out what these Unknown jobs actually are

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b (2).csv")

if len(found) == 0:
    print("❌ Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"✅ Loaded {len(jobs)} jobs from:", FOLDER)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 20 jobs from: /content/drive/MyDrive/MIST 4600/


In [3]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))



country
Germany    1
Spain      3
dtype: int64


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import pandas as pd

# Look for the full 4,000-row dfall_clean.csv first, or fall back to the 20-row sample CSV
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])

Mounted at /content/drive
Loaded 20 jobs from: /content/drive/MyDrive/MIST 4600/sample_rows__3__629d0f3b (2).csv


In [4]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))

country
Germany    1
Spain      3
dtype: int64


In [5]:
import time
import requests
from google import genai
from google.colab import userdata

ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]

def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False

class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

# Answer Sub-Question 1
analyzer = JobPostingAnalyzer(jobs)
jobs = analyzer.mark_missing()

for country in analyzer.countries:
    pct = analyzer.missing_rate(country)
    print(f"{country}: {pct}% missing")

# Average salary by country (Week 10 groupby)
print("\nMean salary by country:")
print(jobs.groupby("country")["salary_mid"].mean())

Germany: 33.3% missing
Switzerland: 100.0% missing
France: 0.0% missing
Spain: 100.0% missing
Netherlands: 0.0% missing
Italy: 0.0% missing
United Kingdom: 0.0% missing
USA: 0.0% missing

Mean salary by country:
country
France                   NaN
Germany                  NaN
Italy                    NaN
Spain              3717.0000
Switzerland              NaN
USA               71761.1375
United Kingdom    43527.4120
Name: salary_mid, dtype: float64


In [8]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import pandas as pd

# 1. Search Google Drive (and current folder) for the 4,000-row dfall_clean.csv
found = glob.glob("dfall_clean.csv")
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

# 2. If dfall_clean.csv is inside archive*.zip on Drive, pandas reads the zip directly
if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "archive*.zip")

# 3. Fallback to the 20-row sample CSV only if neither full file is found
if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find dfall_clean.csv in Google Drive. Please upload dfall_clean.csv to MyDrive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])
    print("Shape:", jobs.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 4000 jobs from: /content/drive/MyDrive/MIST 4600/dfall_clean.csv
Shape: (4000, 20)


In [9]:
# Step 2: Define ALLOWED categories, helper function, JobPostingAnalyzer class, and Sub-Question 1 function
ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]


def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False


class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

    def top_categories(self, country, column_name, n=5):
        sub = self.df[self.df["country"] == country]
        return sub[column_name].value_counts().head(n)


def answer_subquestion_1(analyzer):
    df = analyzer.mark_missing()
    print("--- Sub-Question 1: Missing Category Rate by Country ---")
    for country in analyzer.countries:
        sub = df[df["country"] == country]
        miss = sub[sub["is_missing"] == True]
        pct = analyzer.missing_rate(country)
        print(f"{country}: {pct}% missing ({len(miss)}/{len(sub)})")

    total_miss = len(df[df["is_missing"] == True])
    total_pct = round((total_miss / len(df)) * 100, 1)
    print(f"\nTotal Missing Across All 8 Countries: {total_miss}/{len(df)} ({total_pct}%)")

    print("\n--- Mean Salary by Country (Local Currency) ---")
    print(df.groupby("country")["salary_mid"].mean().round(2))
    return df


analyzer = JobPostingAnalyzer(jobs)
jobs = answer_subquestion_1(analyzer)

--- Sub-Question 1: Missing Category Rate by Country ---
Germany: 86.8% missing (434/500)
Switzerland: 71.0% missing (355/500)
France: 65.0% missing (325/500)
Spain: 64.6% missing (323/500)
Netherlands: 56.2% missing (281/500)
Italy: 31.8% missing (159/500)
United Kingdom: 0.4% missing (2/500)
USA: 0.0% missing (0/500)

Total Missing Across All 8 Countries: 1879/4000 (47.0%)

--- Mean Salary by Country (Local Currency) ---
country
France            123224.47
Germany            23114.74
Italy              19035.11
Netherlands        34016.83
Spain               7019.12
Switzerland        31469.00
USA                74163.67
United Kingdom     39405.11
Name: salary_mid, dtype: float64


In [ ]:
# Step 3: Connect to Gemini via Colab Secrets and classify job titles into a new column
import time
from google import genai
from google.colab import userdata

# Add GOOGLE_API_KEY in Colab Secrets (key icon on left) and turn Notebook access ON
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)


def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
    )
    answer = "other / specialized"
    if response.text:
        answer = response.text.strip().lower()
    time.sleep(4)

    if answer in ALLOWED:
        return answer
    return "other / specialized"


def classify_row(title):
    return get_job_category(title, client)


# Run on a 5-row sample with a 4-second pause to stay under rate limits
sample_size = 5
if len(jobs) < 5:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

print("--- Sub-Question 2: 5 Sample Rows Classified by Gemini (imputed_category) ---")
print(sample_jobs[["country", "title", "category_label", "imputed_category"]])

In [ ]:
# Step 4: Check match rate on already-labeled rows and count jobs uncovered in missing rows
def check_category_accuracy(sample_df):
    labeled = sample_df[sample_df["is_missing"] == False]
    orig_list = []
    for val in labeled["category_label"]:
        orig_list.append(str(val).strip().lower())

    matches = 0
    for i, new_cat in enumerate(labeled["imputed_category"]):
        first_word = new_cat.split(" ")[0]
        if first_word in orig_list[i]:
            matches += 1

    print("--- Sub-Question 2 Validation: Match Rate on Labeled Sample Rows ---")
    print("Labeled rows in sample:", len(labeled))
    if len(labeled) > 0:
        match_pct = round((matches / len(labeled)) * 100, 1)
        print(f"First-word matches: {matches}/{len(labeled)} ({match_pct}%)")

    missing_only = sample_df[sample_df["is_missing"] == True]
    print("\n--- Hidden Jobs Uncovered Inside Missing Sample Rows ---")
    if len(missing_only) == 0:
        print("No missing-category rows landed in this 5-row random sample.")
    else:
        print(missing_only["imputed_category"].value_counts())


check_category_accuracy(sample_jobs)

In [ ]:
# Step 5: Call Frankfurter Currency Web API, convert salaries to USD, compare before/after, and recommend action
import requests


def answer_subquestion_3(full_df, sample_df, analyzer, client):
    # 1. Week 11 Web API: Live EUR exchange rates from Frankfurter API
    url = "https://api.frankfurter.app/latest"
    params = {"from": "EUR", "to": "USD,GBP,CHF"}
    resp = requests.get(url, params=params)
    rates = {"USD": 1.08, "GBP": 0.85, "CHF": 0.94}
    print("--- Week 11 Web API: Live EUR Exchange Rates (Frankfurter API) ---")
    print("Status code:", resp.status_code)
    if resp.status_code == 200:
        rates = resp.json()["rates"]
    print("Live EUR rates:", rates)

    # 2. Convert mean salary per country to USD
    eur_to_usd = rates.get("USD", 1.08)
    eur_to_gbp = rates.get("GBP", 0.85)
    eur_to_chf = rates.get("CHF", 0.94)
    country_means = full_df.groupby("country")["salary_mid"].mean()
    euro_countries = ["Germany", "France", "Spain", "Italy", "Netherlands"]

    print("\n--- Sub-Question 3: Mean Salary Converted to USD ---")
    for country in analyzer.countries:
        if country in country_means.index and pd.notna(country_means[country]):
            local_avg = country_means[country]
            if country == "USA":
                usd_avg = local_avg
            elif country == "United Kingdom":
                usd_avg = (local_avg / eur_to_gbp) * eur_to_usd
            elif country == "Switzerland":
                usd_avg = (local_avg / eur_to_chf) * eur_to_usd
            elif country in euro_countries:
                usd_avg = local_avg * eur_to_usd
            else:
                usd_avg = local_avg
            print(f"{country}: ${round(usd_avg, 0):,.0f} USD")

    # 3. Build final_category column (replace missing tags with Gemini's imputed_category)
    final_list = []
    is_missing_list = list(sample_df["is_missing"])
    orig_cat_list = list(sample_df["category_label"])
    new_cat_list = list(sample_df["imputed_category"])

    for i in range(len(sample_df)):
        if is_missing_list[i] == True:
            final_list.append(new_cat_list[i])
        else:
            final_list.append(str(orig_cat_list[i]).strip().lower())

    sample_df["final_category"] = final_list
    print("\n--- Before vs. After Category Counts (Sample) ---")
    print("Before Imputation (category_label):")
    print(sample_df["category_label"].value_counts())
    print("\nAfter Imputation (final_category):")
    print(sample_df["final_category"].value_counts())

    # 4. Week 12 "\n".join() summary of missing German titles
    de_jobs = full_df[full_df["country"] == "Germany"]
    de_missing = de_jobs[de_jobs["is_missing"] == True]
    if len(de_missing) > 0:
        time.sleep(4)
        titles_text = "\n".join(de_missing.head(10)["title"].astype(str))
        summary = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="In 3 short bullets, what jobs are these and what should a job board do?\n" + titles_text,
        )
        print("\n--- Gemini Business Summary for Unlabeled German Postings ---")
        print(summary.text)

    # 5. Final Business Recommendation
    high_missing = []
    top_country = ""
    top_pct = -1.0
    for country in analyzer.countries:
        pct = analyzer.missing_rate(country)
        if pct > top_pct:
            top_pct = pct
            top_country = country
        if pct >= 15.0:
            high_missing.append(f"{country} ({pct}%)")

    print("\n=== FINAL BUSINESS RECOMMENDATION ===")
    print(f"1. Biggest blind spot: {top_country} at {top_pct}% missing categories.")
    if len(high_missing) > 0:
        print("2. Priority markets (15%+ unlabeled):", ", ".join(high_missing))
    print("3. Action: Run Gemini auto-classification on European job titles at ingestion and target search ads to top unveiled sectors.")

    sample_df.to_csv("classified_jobs.csv", index=False)
    print("Saved classified_jobs.csv")


answer_subquestion_3(jobs, sample_jobs, analyzer, client)

In [7]:
# Make sure GOOGLE_API_KEY is added in Colab Secrets (key icon on left) with Notebook access ON
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    answer = "other / specialized"
    if response.text:
        answer = response.text.strip().lower()
    time.sleep(1)

    if answer in ALLOWED:
        return answer
    return "other / specialized"

def classify_row(title):
    return get_job_category(title, client)

# Run on a 10-row sample so it finishes fast in class
sample_size = 10
if len(jobs) < 10:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

# Spot-check 5 rows
show_n = 5
if len(sample_jobs) < 5:
    show_n = len(sample_jobs)
print(sample_jobs[["country", "title", "category_label", "imputed_category"]].sample(show_n))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
import time
from google import genai
from google.colab import userdata

# Add GOOGLE_API_KEY in Colab Secrets (key icon on left) and turn Notebook access ON
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
    )
    answer = "other / specialized"
    if response.text:
        answer = response.text.strip().lower()
    time.sleep(4)

    if answer in ALLOWED:
        return answer
    return "other / specialized"

def classify_row(title):
    return get_job_category(title, client)

# Run on a 5-row sample with 4-second pause (Week 12 Cheat Sheet) to stay under rate limits
sample_size = 5
if len(jobs) < 5:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

# Show all classified sample rows
print(sample_jobs[["country", "title", "category_label", "imputed_category"]])

In [ ]:
import time
import requests

# Compare Gemini vs existing labels on non-missing rows
labeled = sample_jobs[sample_jobs["is_missing"] == False]
orig_list = []
for val in labeled["category_label"]:
    orig_list.append(str(val).strip().lower())

matches = 0
for i, new_cat in enumerate(labeled["imputed_category"]):
    first_word = new_cat.split(" ")[0]
    if first_word in orig_list[i]:
        matches += 1

if len(labeled) > 0:
    print(f"Match rate: {matches}/{len(labeled)}")

# Week 11 Web API: Live EUR exchange rates
url = "https://api.frankfurter.app/latest"
params = {"from": "EUR", "to": "USD,GBP,CHF"}
resp = requests.get(url, params=params)
if resp.status_code == 200:
    print("Live EUR rates:", resp.json()["rates"])

# Week 12 "\n".join() summary for missing German jobs
de_jobs = jobs[jobs["country"] == "Germany"]
de_missing = de_jobs[de_jobs["is_missing"] == True]
if len(de_missing) > 0:
    time.sleep(4)
    titles_text = "\n".join(de_missing.head(10)["title"].astype(str))
    summary = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="In 3 bullets, what jobs are these and what should a job board do?\n" + titles_text,
    )
    print("\nBusiness Summary:\n", summary.text)

# Save output CSV
sample_jobs.to_csv("classified_jobs.csv", index=False)
print("\nSaved classified_jobs.csv")